# Exploration & jointures des données Gold

Trois niveaux de jointure :
1. **National annuel** — GES Citepa + empreinte carbone
2. **Département × année** — GES communes agrégé + catastrophes + incendies (+ SWI si dispo)
3. **Commune × année** — catastrophes + incendies à la maille commune

In [1]:
import os
from pathlib import Path

import pandas as pd

# Remonte jusqu'à la racine du projet
ROOT = Path(os.getcwd())
while ROOT.name != "hackaton" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
GOLD = ROOT / "data" / "gold"
print("ROOT:", ROOT)
print("Fichiers gold:", [f.name for f in sorted(GOLD.glob("*.parquet"))])

ROOT: /home/seb/cours/hackaton
Fichiers gold: ['cata.parquet', 'climat_metropole.parquet', 'empreinte_carbone.parquet', 'ges_citepa.parquet', 'ges_communes.parquet', 'incendies.parquet']


## 1. Chargement

In [2]:
cata         = pd.read_parquet(GOLD / "cata.parquet")
climat       = pd.read_parquet(GOLD / "climat_metropole.parquet")
empreinte    = pd.read_parquet(GOLD / "empreinte_carbone.parquet")
ges_citepa   = pd.read_parquet(GOLD / "ges_citepa.parquet")
ges_communes = pd.read_parquet(GOLD / "ges_communes.parquet")
incendies    = pd.read_parquet(GOLD / "incendies.parquet")

swi_dept_path = GOLD / "swi_par_departement.parquet"
swi_dept = pd.read_parquet(swi_dept_path) if swi_dept_path.exists() else None

for name, df in [
    ("cata", cata), ("climat", climat), ("empreinte", empreinte),
    ("ges_citepa", ges_citepa), ("ges_communes", ges_communes),
    ("incendies", incendies),
]:
    print(f"{name:20s}  {len(df):>9,} lignes  {list(df.columns)}")
if swi_dept is not None:
    print(f"{'swi_dept':20s}  {len(swi_dept):>9,} lignes  {list(swi_dept.columns)}")

cata                    149,169 lignes  ['code_insee', 'code_dept', 'commune', 'peril', 'date_debut', 'date_fin', 'duree_jours']
climat                1,175,976 lignes  ['date', 'num_poste', 'nom_usuel', 'latitude', 'longitude', 'altitude', 'tn', 'q_hom_tn', 'tx', 'q_hom_tx', 'rr', 'q_hom_rr']
empreinte                    35 lignes  ['annee', 'empreinte_par_personne_tco2eq', 'empreinte_totale_mtco2eq', 'emissions_directes_menages_mtco2eq']
ges_citepa                  560 lignes  ['annee', 'substance', 'valeur', 'unite', 'utcatf_inclus']
ges_communes            730,128 lignes  ['annee', 'geocode_commune', 'libelle_commune', 'secteur', 'valeur']
incendies               140,248 lignes  ['annee', 'numero', 'departement', 'code_insee', 'commune', 'date_alerte', 'surface_totale_m2', 'surface_foret_m2', 'surface_maquis_m2', 'surface_autres_naturelles_m2', 'surface_agricole_m2', 'surface_autres_m2', 'surface_autres_boisees_m2', 'surface_non_boisee_naturelle_m2', 'surface_non_boisee_artificiell

## 2. Panel national annuel

Jointure `ges_citepa` (pivotée par substance) × `empreinte_carbone` sur `annee`.

In [3]:
# Total CO2e hors UTCATF
ges_total = (
    ges_citepa[~ges_citepa["utcatf_inclus"]]
    .groupby("annee", as_index=False)
    .agg(ges_total_co2e=("valeur", "sum"))
)

# Pivot : une colonne par substance
ges_pivot = (
    ges_citepa[~ges_citepa["utcatf_inclus"]]
    .pivot_table(index="annee", columns="substance", values="valeur", aggfunc="sum")
    .reset_index()
)
ges_pivot.columns.name = None

national = (
    ges_total
    .merge(ges_pivot, on="annee", how="outer")
    .merge(empreinte, on="annee", how="outer")
    .sort_values("annee")
    .reset_index(drop=True)
)

print(f"shape: {national.shape}   années: {national['annee'].min()}–{national['annee'].max()}")
national.head()

shape: (35, 13)   années: 1990–2024


,annee,ges_total_co2e,Dioxyde de carbone (CO2),Hexafluorure de soufre (SF6),Hydrofluorocarbures (HFC),Méthane (CH4),Perfluorocarbures (PFC),Protoxyde d'azote (N2O),Total gaz à effet de serre (CO2e),Trifluorure d'azote (NF3),empreinte_par_personne_tco2eq,empreinte_totale_mtco2eq,emissions_directes_menages_mtco2eq
0,1990,147709.390781,399.810741,2232.648613,4226.205294,81876.287431,4684.556585,53727.881496,546.573747,15.426873,12.068,702.062428,130.420389
1,1991,148425.284711,424.480081,2297.875024,4850.628979,82183.494173,4344.339671,53735.588245,571.908977,16.969561,12.028,703.138442,138.834628
2,1992,148089.005730,414.106608,2338.405035,4131.277145,82032.369257,4428.821039,54164.139843,561.220287,18.666517,11.791,692.607168,140.393552
3,1993,145676.833875,394.901901,2379.568639,2423.470952,82391.933687,4363.196555,53163.584783,539.644189,20.533168,10.952,645.992235,138.802212
4,1994,143978.200274,388.180589,2533.013991,1498.467540,82299.389766,3898.604793,52806.717741,531.239369,22.586485,10.828,641.012151,134.935370


## 3. Panel département × année

Sources :
- **GES communes** → somme par dept + année, pivot par secteur
- **Catastrophes naturelles** → nb événements, durée, nb communes touchées
- **Incendies** → nb feux, surface totale
- **SWI** → humidité des sols moyenne annuelle (si disponible)

In [4]:
def extract_code_dept(geocode: pd.Series) -> pd.Series:
    """Extrait le code département depuis un code INSEE commune à 5 caractères."""
    s = geocode.astype(str).str.strip()
    # Corse : codes commençant par 2A ou 2B
    corse = s.str.upper().str.startswith(("2A", "2B"))
    dept = s.str[:2]
    dept = dept.where(corse, dept.str.lstrip("0").str.zfill(2))
    return dept

ges_c = ges_communes.copy()
ges_c["code_dept"] = extract_code_dept(ges_c["geocode_commune"])

ges_dept = (
    ges_c.groupby(["code_dept", "annee", "secteur"], as_index=False)
    .agg(valeur=("valeur", "sum"))
    .pivot_table(index=["code_dept", "annee"], columns="secteur", values="valeur", aggfunc="sum")
    .reset_index()
)
ges_dept.columns.name = None
secteur_cols = [c for c in ges_dept.columns if c not in ("code_dept", "annee")]
ges_dept = ges_dept.rename(columns={c: "ges_" + c.lower().replace(" ", "_") for c in secteur_cols})
ges_cols = [c for c in ges_dept.columns if c.startswith("ges_")]
ges_dept["ges_total"] = ges_dept[ges_cols].sum(axis=1)

print(f"GES dept : {ges_dept.shape}  —  années : {sorted(ges_dept['annee'].unique())}")
ges_dept.head(3)

GES dept : (286, 10)  —  années : [np.int64(2016), np.int64(2018), np.int64(2021)]


,code_dept,annee,ges_agriculture,ges_déchets,ges_energie,ges_industrie_hors_energie,ges_résidentiel,ges_tertiaire,ges_transports,ges_total
0,01,2016,821093.158457,264449.406039,119155.359262,1.004964e+06,542931.383929,389992.856146,1.661518e+06,4.804104e+06
1,01,2018,806834.139425,90374.389309,110879.007736,8.214525e+05,518703.312343,378733.705014,1.621734e+06,4.348712e+06
2,01,2021,659108.704690,160910.910042,26405.392303,7.006119e+05,502594.420927,321627.324416,1.372502e+06,3.743761e+06


In [6]:
cata_dept = (
    cata.assign(annee=cata["date_debut"].dt.year)
    .groupby(["code_dept", "annee"], as_index=False)
    .agg(
        cata_nb_evenements=("peril", "count"),
        cata_nb_communes=("code_insee", "nunique"),
        cata_duree_totale_jours=("duree_jours", "sum"),
        cata_nb_perils_distincts=("peril", "nunique"),
    )
)
print(f"Cata dept : {cata_dept.shape}  —  {cata_dept['annee'].min()}–{cata_dept['annee'].max()}")
cata_dept.head(3)

Cata dept : (2742, 6)  —  1982–2015


,code_dept,annee,cata_nb_evenements,cata_nb_communes,cata_duree_totale_jours,cata_nb_perils_distincts
0,1,1982,2,2,46,1
1,1,1983,17,16,28,2
2,1,1984,3,3,0,1


In [7]:
inc_dept = (
    incendies.rename(columns={"departement": "code_dept"})
    .groupby(["code_dept", "annee"], as_index=False)
    .agg(
        inc_nb_feux=("numero", "count"),
        inc_surface_totale_m2=("surface_totale_m2", "sum"),
        inc_surface_foret_m2=("surface_foret_m2", "sum"),
        inc_nb_deces=("nb_deces", "sum"),
    )
)
print(f"Incendies dept : {inc_dept.shape}  —  {inc_dept['annee'].min()}–{inc_dept['annee'].max()}")
inc_dept.head(3)

Incendies dept : (1580, 6)  —  1973–2024


,code_dept,annee,inc_nb_feux,inc_surface_totale_m2,inc_surface_foret_m2,inc_nb_deces
0,01,2006,12,270000,0.0,0.0
1,01,2007,12,165000,5000.0,0.0
2,01,2008,13,267000,0.0,0.0


In [8]:
panel_dept = (
    ges_dept
    .merge(cata_dept, on=["code_dept", "annee"], how="outer")
    .merge(inc_dept,  on=["code_dept", "annee"], how="outer")
)

if swi_dept is not None:
    swi_annual = (
        swi_dept.assign(annee=swi_dept["date"].dt.year)
        .groupby(["code_dept", "annee"], as_index=False)
        .agg(swi_moyen_annuel=("swi_moyen", "mean"))
    )
    panel_dept = panel_dept.merge(swi_annual, on=["code_dept", "annee"], how="outer")

panel_dept = panel_dept.sort_values(["code_dept", "annee"]).reset_index(drop=True)

print(f"Panel département : {panel_dept.shape}")
print(f"Départements      : {panel_dept['code_dept'].nunique()}")
print(f"Années            : {panel_dept['annee'].min()}–{panel_dept['annee'].max()}")
panel_dept.head()

Panel département : (3895, 18)
Départements      : 110
Années            : 1973–2024


,code_dept,annee,ges_agriculture,ges_déchets,ges_energie,ges_industrie_hors_energie,ges_résidentiel,ges_tertiaire,ges_transports,ges_total,cata_nb_evenements,cata_nb_communes,cata_duree_totale_jours,cata_nb_perils_distincts,inc_nb_feux,inc_surface_totale_m2,inc_surface_foret_m2,inc_nb_deces
0,01,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,270000.0,0.0,0.0
1,01,2007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.0,165000.0,5000.0,0.0
2,01,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,267000.0,0.0,0.0
3,01,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,130000.0,30000.0,0.0
4,01,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,3900000.0,1390000.0,0.0


In [9]:
coverage = panel_dept.notna().mean().sort_values(ascending=False).round(3)
print("Taux de remplissage par colonne :")
print(coverage.to_string())

Taux de remplissage par colonne :
code_dept                     1.000
annee                         1.000
cata_nb_perils_distincts      0.704
cata_duree_totale_jours       0.704
cata_nb_communes              0.704
cata_nb_evenements            0.704
inc_nb_deces                  0.406
inc_surface_foret_m2          0.406
inc_surface_totale_m2         0.406
inc_nb_feux                   0.406
ges_total                     0.073
ges_transports                0.073
ges_tertiaire                 0.073
ges_résidentiel               0.073
ges_industrie_hors_energie    0.073
ges_energie                   0.073
ges_déchets                   0.073
ges_agriculture               0.073


## 4. Panel commune × année

Jointure `cata` + `incendies` sur `code_insee` + `annee`.

In [10]:
cata_commune = (
    cata.assign(annee=cata["date_debut"].dt.year)
    .groupby(["code_insee", "code_dept", "commune", "annee"], as_index=False)
    .agg(
        cata_nb_evenements=("peril", "count"),
        cata_duree_totale_jours=("duree_jours", "sum"),
        cata_perils=("peril", lambda x: ",".join(sorted(set(x)))),
    )
)

inc_commune = (
    incendies.groupby(["code_insee", "annee"], as_index=False)
    .agg(
        inc_nb_feux=("numero", "count"),
        inc_surface_totale_m2=("surface_totale_m2", "sum"),
    )
)

panel_commune = (
    cata_commune
    .merge(inc_commune, on=["code_insee", "annee"], how="outer")
    .sort_values(["code_insee", "annee"])
    .reset_index(drop=True)
)

print(f"Panel commune : {panel_commune.shape}")
print(f"Communes      : {panel_commune['code_insee'].nunique()}")
print(f"Années        : {panel_commune['annee'].min()}–{panel_commune['annee'].max()}")
panel_commune.head()

Panel commune : (189435, 9)
Communes      : 37287
Années        : 1973–2024


,code_insee,code_dept,commune,annee,cata_nb_evenements,cata_duree_totale_jours,cata_perils,inc_nb_feux,inc_surface_totale_m2
0,01014,NaN,NaN,2007,NaN,NaN,NaN,2.0,50000.0
1,01015,NaN,NaN,2007,NaN,NaN,NaN,1.0,20000.0
2,01017,NaN,NaN,2015,NaN,NaN,NaN,1.0,100000.0
3,01032,NaN,NaN,2006,NaN,NaN,NaN,1.0,10000.0
4,01034,NaN,NaN,2006,NaN,NaN,NaN,1.0,30000.0


## 5. Export

In [11]:
national.to_parquet(GOLD / "panel_national.parquet", index=False, engine="pyarrow")
panel_dept.to_parquet(GOLD / "panel_departement.parquet", index=False, engine="pyarrow")
panel_commune.to_parquet(GOLD / "panel_commune.parquet", index=False, engine="pyarrow")

print("Exports :")
print(f"  panel_national.parquet     {national.shape}")
print(f"  panel_departement.parquet  {panel_dept.shape}")
print(f"  panel_commune.parquet      {panel_commune.shape}")

Exports :
  panel_national.parquet     (35, 13)
  panel_departement.parquet  (3895, 18)
  panel_commune.parquet      (189435, 9)
